# Feature Selection

This notebook will provide the justifcations for feature selection.

In [12]:
# Packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# First 5 rows of dataset
df.show(5)

time,src_user,dest_user,src_comp,dest_comp,auth_type,logon_type,auth_orientation,outcome
i64,str,str,str,str,str,str,str,str
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
1,"""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
1,"""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
1,"""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


In [ ]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

In [ ]:
# Features dataframe
features = (
    df
    .with_columns(
        bucket=pl.col("time") // 3600,
        hour_of_day=(pl.col("time") // 3600) % 24,
        is_failure=(pl.col("outcome") != "Success").cast(pl.Int8),
    )
    .group_by(["src_user", "bucket"])
    .agg(
        hour_of_day=pl.col("hour_of_day").first(),
        n_events=pl.len(),
        n_failures=pl.col("is_failure").sum(),
        failure_ratio=pl.col("is_failure").mean(),
        n_distinct_dest=pl.col("dest_comp").n_unique(),
        n_distinct_src=pl.col("src_comp").n_unique(),
        n_distinct_auth_type=pl.col("auth_type").n_unique(),
        n_distinct_logon_type=pl.col("logon_type").n_unique(),
    )
    .collect()
)

In [11]:
features.show(100)

src_user,bucket,hour_of_day,n_events,n_failures,failure_ratio,n_distinct_dest,n_distinct_src,n_distinct_auth_type,n_distinct_logon_type
str,i64,i64,u64,i64,f64,u64,u64,u64,u64
"""U4769@DOM1""",1001,17,33,0,0.0,6,7,2,2
"""U8984@DOM1""",909,21,6,0,0.0,2,2,2,2
"""U8911@DOM1""",417,9,78,0,0.0,3,4,3,1
"""U1439@C2198""",1297,1,3,0,0.0,1,1,1,2
"""U2395@DOM1""",1024,16,40,0,0.0,7,4,2,3
"""U1096@DOM1""",465,9,31,0,0.0,8,6,2,2
"""U6455@DOM1""",606,6,20,0,0.0,3,4,2,2
"""U5010@DOM1""",415,7,18,0,0.0,3,4,2,1
"""U9191@DOM1""",1165,13,53,1,0.018868,8,6,3,3


In [13]:
features = features.with_columns(
    hod_sin=(2 * np.pi * pl.col("hour_of_day") / 24).sin(),
    hod_cos=(2 * np.pi * pl.col("hour_of_day") / 24).cos(),
)

In [14]:
features.show(10)

src_user,bucket,hour_of_day,n_events,n_failures,failure_ratio,n_distinct_dest,n_distinct_src,n_distinct_auth_type,n_distinct_logon_type,hod_sin,hod_cos
str,i64,i64,u64,i64,f64,u64,u64,u64,u64,f64,f64
"""U4769@DOM1""",1001,17,33,0,0.0,6,7,2,2,-0.965926,-0.258819
"""U8984@DOM1""",909,21,6,0,0.0,2,2,2,2,-0.707107,0.707107
"""U8911@DOM1""",417,9,78,0,0.0,3,4,3,1,0.707107,-0.707107
"""U1439@C2198""",1297,1,3,0,0.0,1,1,1,2,0.258819,0.965926
"""U2395@DOM1""",1024,16,40,0,0.0,7,4,2,3,-0.866025,-0.5
"""U1096@DOM1""",465,9,31,0,0.0,8,6,2,2,0.707107,-0.707107
"""U6455@DOM1""",606,6,20,0,0.0,3,4,2,2,1.0,6.1232e-17
"""U5010@DOM1""",415,7,18,0,0.0,3,4,2,1,0.965926,-0.258819
"""U9191@DOM1""",1165,13,53,1,0.018868,8,6,3,3,-0.258819,-0.965926
